In [1]:
import pandas as pd
import numpy as np

# Carregar o dataset
df = pd.read_parquet("mimic_com_labels_exames_fill.parquet")

# Ordenar por paciente, internação e tempo
df = df.sort_values(["subject_id", "stay_id", "charttime"])

# Preencher valores faltantes dentro de cada paciente/internação
for coluna in df.columns:

    # Ignora colunas identificadoras e datas
    if coluna in ["subject_id", "stay_id", "janela_index", "inicio_janela", "charttime"]:
        continue

    # Preenchimento temporal (último valor conhecido -> próximo valor conhecido)
    df[coluna] = (
        df.groupby(["subject_id", "stay_id"])[coluna]
        .transform(lambda x: x.ffill().bfill())
    )

# Substituir strings vazias por NaN
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)

# Preencher qualquer NaN restante
for coluna in df.columns:

    if df[coluna].isnull().sum() == 0:
        continue

    if pd.api.types.is_numeric_dtype(df[coluna]):
        df[coluna].fillna(df[coluna].median(), inplace=True)
    else:
        moda = df[coluna].mode(dropna=True)
        if len(moda) > 0:
            df[coluna].fillna(moda.iloc[0], inplace=True)
        else:
            df[coluna].fillna("", inplace=True)

# Conferência
print("Total de valores nulos:", df.isnull().sum().sum())
print("Total de strings vazias:", (df.astype(str).eq("")).sum().sum())

# Salvar
df.to_parquet("mimic_com_labels_exames_fill.parquet", index=False)

print("Arquivo salvo com sucesso!")

C:\Users\walter.cavalcante\AppData\Local\Temp\ipykernel_15108\2990998702.py:33: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[coluna].fillna(df[coluna].median(), inplace=True)
C:\Users\walter.cavalcante\AppData\Local\Temp\ipykernel_15108\2990998702.py:33: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values alwa

Total de valores nulos: 0
Total de strings vazias: 0
Arquivo salvo com sucesso!
